# BIRD-lite: Building Image Line Detection Pipeline

Line segment extraction from building drawings using classical computer vision.

## Requirements
- Classical CV: Canny edge detection + HoughLinesP
- JSON output format
- Robustness testing: rotation invariance (0-90 degrees)

In [1]:
import subprocess, sys, json
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
INPUT_DIR.mkdir(exist_ok=True, parents=True)

IMG_SIZE, LINE_THICKNESS = 512, 2
LINE_COLOR = (255, 255, 255)
CANNY_LOW, CANNY_HIGH = 50, 150
RHO, THETA = 1, np.pi / 180
THRESHOLD, MIN_LINE_LENGTH, MAX_LINE_GAP = 50, 30, 10
MERGE_THRESHOLD, ANGLE_THRESHOLD = 15, 10
ROTATION_ANGLES = [0, 15, 30, 45, 60, 75, 90]

print('Configuration loaded')

Configuration loaded


## Step 1: Generate Synthetic Datasets (10 building layouts)

In [2]:
def create_simple_building_1():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    pt1, pt2 = (80, 100), (430, 400)
    cv2.rectangle(img, pt1, pt2, LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': pt1, 'end': (430, 100), 'type': 'wall'},
                  {'start': (430, 100), 'end': (430, 400), 'type': 'wall'},
                  {'start': (430, 400), 'end': (80, 400), 'type': 'wall'},
                  {'start': (80, 400), 'end': (80, 100), 'type': 'wall'}])
    cv2.line(img, (80, 250), (430, 250), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (80, 250), 'end': (430, 250), 'type': 'divider'})
    cv2.line(img, (200, 100), (200, 250), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (200, 100), 'end': (200, 250), 'type': 'divider'})
    cv2.line(img, (320, 100), (320, 250), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (320, 100), 'end': (320, 250), 'type': 'divider'})
    return img, lines

def create_simple_building_2():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    pts = [(100, 150), (400, 150), (400, 280), (100, 280), (100, 150)]
    for i in range(len(pts)-1):
        cv2.line(img, pts[i], pts[i+1], LINE_COLOR, LINE_THICKNESS)
        lines.append({'start': pts[i], 'end': pts[i+1], 'type': 'wall'})
    pts2 = [(250, 280), (370, 280), (370, 400), (250, 400), (250, 280)]
    for i in range(len(pts2)-1):
        cv2.line(img, pts2[i], pts2[i+1], LINE_COLOR, LINE_THICKNESS)
        lines.append({'start': pts2[i], 'end': pts2[i+1], 'type': 'wall'})
    return img, lines

def create_simple_building_3():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    cv2.rectangle(img, (100, 120), (400, 380), LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': (100, 120), 'end': (400, 120), 'type': 'wall'},
                  {'start': (400, 120), 'end': (400, 380), 'type': 'wall'},
                  {'start': (400, 380), 'end': (100, 380), 'type': 'wall'},
                  {'start': (100, 380), 'end': (100, 120), 'type': 'wall'}])
    cv2.line(img, (250, 120), (250, 380), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (250, 120), 'end': (250, 380), 'type': 'divider'})
    cv2.line(img, (100, 250), (400, 250), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (100, 250), 'end': (400, 250), 'type': 'divider'})
    return img, lines

def create_simple_building_4():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    cv2.rectangle(img, (120, 140), (390, 360), LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': (120, 140), 'end': (390, 140), 'type': 'wall'},
                  {'start': (390, 140), 'end': (390, 360), 'type': 'wall'},
                  {'start': (390, 360), 'end': (120, 360), 'type': 'wall'},
                  {'start': (120, 360), 'end': (120, 140), 'type': 'wall'}])
    for x1, x2 in [(150, 190), (310, 360)]:
        cv2.rectangle(img, (x1, 150), (x2, 180), LINE_COLOR, LINE_THICKNESS)
        lines.extend([{'start': (x1, 150), 'end': (x2, 150), 'type': 'window'},
                      {'start': (x2, 150), 'end': (x2, 180), 'type': 'window'},
                      {'start': (x2, 180), 'end': (x1, 180), 'type': 'window'},
                      {'start': (x1, 180), 'end': (x1, 150), 'type': 'window'}])
    return img, lines

def create_simple_building_5():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    cv2.rectangle(img, (130, 240), (380, 360), LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': (130, 240), 'end': (380, 240), 'type': 'wall'},
                  {'start': (380, 240), 'end': (380, 360), 'type': 'wall'},
                  {'start': (380, 360), 'end': (130, 360), 'type': 'wall'},
                  {'start': (130, 360), 'end': (130, 240), 'type': 'wall'}])
    cv2.line(img, (130, 240), (255, 120), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (130, 240), 'end': (255, 120), 'type': 'roof'})
    cv2.line(img, (255, 120), (380, 240), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (255, 120), 'end': (380, 240), 'type': 'roof'})
    return img, lines

def create_medium_building_6():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    pts = [(90, 100), (280, 80), (420, 150), (430, 380), (200, 420), (80, 350), (90, 100)]
    for i in range(len(pts)-1):
        cv2.line(img, pts[i], pts[i+1], LINE_COLOR, LINE_THICKNESS)
        lines.append({'start': pts[i], 'end': pts[i+1], 'type': 'wall'})
    cv2.line(img, (200, 100), (200, 300), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (200, 100), 'end': (200, 300), 'type': 'divider'})
    cv2.line(img, (100, 250), (350, 250), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (100, 250), 'end': (350, 250), 'type': 'divider'})
    return img, lines

def create_medium_building_7():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    cv2.rectangle(img, (80, 90), (430, 420), LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': (80, 90), 'end': (430, 90), 'type': 'wall'},
                  {'start': (430, 90), 'end': (430, 420), 'type': 'wall'},
                  {'start': (430, 420), 'end': (80, 420), 'type': 'wall'},
                  {'start': (80, 420), 'end': (80, 90), 'type': 'wall'}])
    cv2.rectangle(img, (180, 180), (330, 330), LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': (180, 180), 'end': (330, 180), 'type': 'courtyard'},
                  {'start': (330, 180), 'end': (330, 330), 'type': 'courtyard'},
                  {'start': (330, 330), 'end': (180, 330), 'type': 'courtyard'},
                  {'start': (180, 330), 'end': (180, 180), 'type': 'courtyard'}])
    cv2.line(img, (180, 200), (180, 240), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (180, 200), 'end': (180, 240), 'type': 'passage'})
    return img, lines

def create_medium_building_8():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    for i in range(3):
        y_start, x_start = 120 + i * 80, 120 + i * 80
        x_end, y_end = x_start + 150, y_start + 150
        cv2.rectangle(img, (x_start, y_start), (x_end, y_end), LINE_COLOR, LINE_THICKNESS)
        lines.extend([{'start': (x_start, y_start), 'end': (x_end, y_start), 'type': 'wall'},
                      {'start': (x_end, y_start), 'end': (x_end, y_end), 'type': 'wall'},
                      {'start': (x_end, y_end), 'end': (x_start, y_end), 'type': 'wall'},
                      {'start': (x_start, y_end), 'end': (x_start, y_start), 'type': 'wall'}])
    return img, lines

def create_medium_building_9():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    cv2.rectangle(img, (200, 150), (310, 360), LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': (200, 150), 'end': (310, 150), 'type': 'wall'},
                  {'start': (310, 150), 'end': (310, 360), 'type': 'wall'},
                  {'start': (310, 360), 'end': (200, 360), 'type': 'wall'},
                  {'start': (200, 360), 'end': (200, 150), 'type': 'wall'}])
    for x1, x2 in [(90, 200), (310, 420)]:
        cv2.rectangle(img, (x1, 220), (x2, 290), LINE_COLOR, LINE_THICKNESS)
        lines.extend([{'start': (x1, 220), 'end': (x2, 220), 'type': 'wall'},
                      {'start': (x2, 220), 'end': (x2, 290), 'type': 'wall'},
                      {'start': (x2, 290), 'end': (x1, 290), 'type': 'wall'},
                      {'start': (x1, 290), 'end': (x1, 220), 'type': 'wall'}])
    return img, lines

def create_medium_building_10():
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    lines = []
    cv2.rectangle(img, (100, 110), (410, 390), LINE_COLOR, LINE_THICKNESS)
    lines.extend([{'start': (100, 110), 'end': (410, 110), 'type': 'wall'},
                  {'start': (410, 110), 'end': (410, 390), 'type': 'wall'},
                  {'start': (410, 390), 'end': (100, 390), 'type': 'wall'},
                  {'start': (100, 390), 'end': (100, 110), 'type': 'wall'}])
    cv2.line(img, (200, 110), (200, 250), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (200, 110), 'end': (200, 250), 'type': 'divider'})
    cv2.line(img, (310, 110), (310, 350), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (310, 110), 'end': (310, 350), 'type': 'divider'})
    cv2.line(img, (100, 250), (200, 250), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (100, 250), 'end': (200, 250), 'type': 'divider'})
    cv2.line(img, (200, 300), (310, 300), LINE_COLOR, LINE_THICKNESS)
    lines.append({'start': (200, 300), 'end': (310, 300), 'type': 'divider'})
    return img, lines

builders = [
    ('building_01_simple', create_simple_building_1),
    ('building_02_simple', create_simple_building_2),
    ('building_03_simple', create_simple_building_3),
    ('building_04_simple', create_simple_building_4),
    ('building_05_simple', create_simple_building_5),
    ('building_06_medium', create_medium_building_6),
    ('building_07_medium', create_medium_building_7),
    ('building_08_medium', create_medium_building_8),
    ('building_09_medium', create_medium_building_9),
    ('building_10_medium', create_medium_building_10),
]

ground_truth = {}
for name, builder_func in builders:
    img, lines = builder_func()
    filepath = INPUT_DIR / f'{name}.png'
    cv2.imwrite(str(filepath), img)
    ground_truth[name] = {'image': str(filepath), 'size': (IMG_SIZE, IMG_SIZE), 'lines': lines}

gt_filepath = INPUT_DIR / 'ground_truth.json'
with open(gt_filepath, 'w') as f:
    json.dump(ground_truth, f, indent=2)

print(f'Generated {len(builders)} building layouts')

Generated 10 building layouts


## Step 2: Core Line Detection Functions

In [ ]:
def normalize_line(pt1, pt2):
    (x1, y1), (x2, y2) = pt1, pt2
    if y1 > y2 or (y1 == y2 and x1 > x2):
        return (x2, y2), (x1, y1)
    return (x1, y1), (x2, y2)

def line_distance(pt1, pt2):
    return np.sqrt((pt1[0] - pt2[0])**2 + (pt1[1] - pt2[1])**2)

def line_angle(pt1, pt2):
    dx, dy = pt2[0] - pt1[0], pt2[1] - pt1[1]
    angle = np.arctan2(dy, dx) * 180 / np.pi
    return (angle if angle >= 0 else angle + 180) % 180

def line_similarity(line1, line2, dist_threshold=MERGE_THRESHOLD, angle_threshold=ANGLE_THRESHOLD):
    pt1_start, pt1_end = line1
    pt2_start, pt2_end = line2
    d_ss = line_distance(pt1_start, pt2_start)
    d_se = line_distance(pt1_start, pt2_end)
    d_es = line_distance(pt1_end, pt2_start)
    d_ee = line_distance(pt1_end, pt2_end)
    min_dist = min(d_ss, d_se, d_es, d_ee)
    angle1 = line_angle(pt1_start, pt1_end)
    angle2 = line_angle(pt2_start, pt2_end)
    angle_diff = min(abs(angle1 - angle2), 180 - abs(angle1 - angle2))
    return min_dist < dist_threshold and angle_diff < angle_threshold

def merge_similar(lines, merge_threshold=MERGE_THRESHOLD, angle_threshold=ANGLE_THRESHOLD):
    if len(lines) < 2:
        return lines
    lines = list(set(lines))
    merged, used = [], set()
    for i, line1 in enumerate(lines):
        if i in used:
            continue
        candidates = [line1]
        for j, line2 in enumerate(lines):
            if j <= i or j in used:
                continue
            if line_similarity(line1, line2, merge_threshold, angle_threshold):
                candidates.append(line2)
                used.add(j)
        if len(candidates) > 1:
            starts = [c[0] for c in candidates]
            ends = [c[1] for c in candidates]
            avg_start = (int(np.mean([s[0] for s in starts])), int(np.mean([s[1] for s in starts])))
            avg_end = (int(np.mean([e[0] for e in ends])), int(np.mean([e[1] for e in ends])))
            merged.append(normalize_line(avg_start, avg_end))
        else:
            merged.append(line1)
        used.add(i)
    return merged

print('Core functions loaded')

Core functions loaded


## V1: Baseline Algorithm (Morphology + Merging)

In [ ]:
def detect_lines_v1(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, None, None
    original = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    img = cv2.GaussianBlur(img, (3, 3), 0)
    edges = cv2.Canny(img, CANNY_LOW, CANNY_HIGH)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=1)
    lines_raw = cv2.HoughLinesP(edges, rho=RHO, theta=THETA, threshold=THRESHOLD,
                                minLineLength=MIN_LINE_LENGTH, maxLineGap=MAX_LINE_GAP)
    if lines_raw is None:
        return [], original, edges
    lines = [normalize_line((line[0][0], line[0][1]), (line[0][2], line[0][3])) for line in lines_raw]
    lines = merge_similar(lines, MERGE_THRESHOLD, ANGLE_THRESHOLD)
    return list(set(lines)), original, edges

print('V1 defined')

V1 defined


## V2: Collinear Merging (Higher Threshold)

In [5]:
def points_on_line(p1, p2, p3, threshold=5):
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    if x2 == x1:
        dist = abs(x3 - x1)
    else:
        m = (y2 - y1) / (x2 - x1)
        dist = abs(m * (x3 - x1) - (y3 - y1)) / np.sqrt(m**2 + 1)
    return dist < threshold

def merge_collinear_lines_v2(lines_raw, dist_threshold=8):
    if lines_raw is None or len(lines_raw) == 0:
        return []
    lines = []
    for line in lines_raw:
        x1, y1, x2, y2 = line[0]
        lines.append(((x1, y1), (x2, y2)))
    if len(lines) < 2:
        return lines
    merged = []
    used = set()
    for i, line1 in enumerate(lines):
        if i in used:
            continue
        (x1_a, y1_a), (x1_b, y1_b) = line1
        all_points = [(x1_a, y1_a), (x1_b, y1_b)]
        for j, line2 in enumerate(lines):
            if j <= i or j in used:
                continue
            (x2_a, y2_a), (x2_b, y2_b) = line2
            if (points_on_line((x1_a, y1_a), (x1_b, y1_b), (x2_a, y2_a), dist_threshold) and
                points_on_line((x1_a, y1_a), (x1_b, y1_b), (x2_b, y2_b), dist_threshold)):
                all_points.extend([(x2_a, y2_a), (x2_b, y2_b)])
                used.add(j)
        xs = [p[0] for p in all_points]
        ys = [p[1] for p in all_points]
        merged_line = ((min(xs), min(ys)), (max(xs), max(ys)))
        merged.append(merged_line)
        used.add(i)
    return merged

def detect_lines_v2(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, None, None
    original = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    edges = cv2.Canny(img, CANNY_LOW, CANNY_HIGH)
    lines_raw = cv2.HoughLinesP(edges, rho=RHO, theta=THETA, threshold=100,
                                minLineLength=MIN_LINE_LENGTH, maxLineGap=MAX_LINE_GAP)
    if lines_raw is None:
        return [], original, edges
    lines = merge_collinear_lines_v2(lines_raw, dist_threshold=8)
    return lines, original, edges

print('V2 defined')

V2 defined


## V3: Hough Accumulator Space

In [ ]:
def rho_theta_to_xy(rho, theta, height, width):
    a = np.cos(theta)
    b = np.sin(theta)
    x0 = a * rho
    y0 = b * rho
    x1 = int(x0 + 1000 * (-b))
    y1 = int(y0 + 1000 * a)
    x2 = int(x0 - 1000 * (-b))
    y2 = int(y0 - 1000 * a)
    x1 = max(0, min(width - 1, x1))
    y1 = max(0, min(height - 1, y1))
    x2 = max(0, min(width - 1, x2))
    y2 = max(0, min(height - 1, y2))
    return (x1, y1), (x2, y2)

def hough_space_merge(lines, rho_threshold=10, theta_threshold=np.pi/36):
    if len(lines) < 2:
        return lines
    merged = []
    used = set()
    for i, (rho1, theta1) in enumerate(lines):
        if i in used:
            continue
        group_rhos = [rho1]
        group_thetas = [theta1]
        for j, (rho2, theta2) in enumerate(lines):
            if j <= i or j in used:
                continue
            theta_diff = abs(theta1 - theta2)
            theta_diff = min(theta_diff, np.pi - theta_diff)
            if abs(rho1 - rho2) < rho_threshold and theta_diff < theta_threshold:
                group_rhos.append(rho2)
                group_thetas.append(theta2)
                used.add(j)
        avg_rho = np.mean(group_rhos)
        avg_theta = np.mean(group_thetas)
        merged.append((avg_rho, avg_theta))
        used.add(i)
    return merged

def detect_lines_v3(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, None, None
    original = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    edges = cv2.Canny(img, CANNY_LOW, CANNY_HIGH)
    lines_hough = cv2.HoughLines(edges, rho=1, theta=np.pi/180, threshold=40)
    if lines_hough is None:
        return [], original, edges
    rho_theta_lines = [(line[0][0], line[0][1]) for line in lines_hough]
    merged_rho_theta = hough_space_merge(rho_theta_lines, rho_threshold=10, theta_threshold=np.pi/36)
    h, w = img.shape[:2]
    lines_xy = [rho_theta_to_xy(rho, theta, h, w) for rho, theta in merged_rho_theta]
    return lines_xy, original, edges

print('V3 defined')

V3 defined


## V4: Line Extraction & Resizing

Measurement-based proportioning for rotation invariance

In [ ]:
def Image_max_threshold(edges, percentile=90):
    if edges is None or edges.size == 0:
        return edges
    threshold_value = np.percentile(edges, percentile)
    return np.where(edges > threshold_value, edges, 0).astype(np.uint8)

def extract_point_cloud(thresholded_img):
    points = np.argwhere(thresholded_img > 0)
    return points if len(points) > 0 else np.array([])

def fit_point_cloud(points):
    if len(points) < 4:
        return []
    lines_list = []
    used = set()
    for seed_idx in range(len(points)):
        if seed_idx in used:
            continue
        seed_point = points[seed_idx]
        cluster = [seed_point]
        used.add(seed_idx)
        for j in range(seed_idx + 1, len(points)):
            if j in used:
                continue
            test_point = points[j]
            dist = np.sqrt((seed_point[0] - test_point[0])**2 + (seed_point[1] - test_point[1])**2)
            if dist < 20:
                cluster.append(test_point)
                used.add(j)
        if len(cluster) >= 3:
            cluster = np.array(cluster)
            y_coords = cluster[:, 0]
            x_coords = cluster[:, 1]
            if len(np.unique(x_coords)) > 1:
                try:
                    coeffs = np.polyfit(x_coords, y_coords, 1)
                    x_min, x_max = np.min(x_coords), np.max(x_coords)
                    y_min = coeffs[0] * x_min + coeffs[1]
                    y_max = coeffs[0] * x_max + coeffs[1]
                    lines_list.append(((int(x_min), int(y_min)), (int(x_max), int(y_max))))
                except:
                    continue
            else:
                x_val = int(np.mean(x_coords))
                y_min, y_max = int(np.min(y_coords)), int(np.max(y_coords))
                lines_list.append(((x_val, y_min), (x_val, y_max)))
    return lines_list

def extract_measurements_v4(img_shape):
    h, w = img_shape
    return w * 0.8, h * 0.8

def proportion_lines(lines, max_x, max_y, img_shape):
    if len(lines) == 0:
        return lines
    h, w = img_shape
    scale_x = max_x / w if w > 0 else 1.0
    scale_y = max_y / h if h > 0 else 1.0
    proportioned = []
    for (x1, y1), (x2, y2) in lines:
        x1_scaled = int(x1 * scale_x)
        y1_scaled = int(y1 * scale_y)
        x2_scaled = int(x2 * scale_x)
        y2_scaled = int(y2 * scale_y)
        proportioned.append(((x1_scaled, y1_scaled), (x2_scaled, y2_scaled)))
    return proportioned

def detect_lines_v4(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, None, None
    original = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    edges = cv2.Canny(img, CANNY_LOW, CANNY_HIGH)
    thresholded = Image_max_threshold(edges, percentile=90)
    point_cloud = extract_point_cloud(thresholded)
    if len(point_cloud) == 0:
        return [], original, edges
    lines = fit_point_cloud(point_cloud)
    if len(lines) == 0:
        return [], original, edges
    max_x, max_y = extract_measurements_v4(img.shape)
    proportioned_lines = proportion_lines(lines, max_x, max_y, img.shape)
    return proportioned_lines, original, edges

print('V4 defined')

V4 defined


## Step 3: Rotation Robustness Testing

In [ ]:
image_files = sorted(INPUT_DIR.glob('*.png'))

def rotate_img(image, angle):
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def test_algorithm(algo_name, detect_func, image_files):
    results = {}
    for img_file in image_files:
        img_name = img_file.stem
        original_img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
        original_lines, _, _ = detect_func(img_file)
        original_count = len(original_lines) if original_lines else 0
        results[img_name] = {'original_lines': original_count, 'rotation_angles': {}}
        print(f'{img_name}: Angle | Lines | Change')
        for angle in ROTATION_ANGLES:
            rotated_img = original_img if angle == 0 else rotate_img(original_img, angle)
            edges = cv2.Canny(rotated_img, CANNY_LOW, CANNY_HIGH)
            
            if detect_func == detect_lines_v4:
                thresholded = Image_max_threshold(edges, percentile=90)
                point_cloud = extract_point_cloud(thresholded)
                if len(point_cloud) == 0:
                    detected_count = 0
                else:
                    lines = fit_point_cloud(point_cloud)
                    detected_count = 0 if len(lines) == 0 else len(proportion_lines(lines, *extract_measurements_v4(rotated_img.shape), rotated_img.shape))
            elif detect_func == detect_lines_v3:
                lines_hough = cv2.HoughLines(edges, rho=1, theta=np.pi/180, threshold=40)
                detected_count = 0 if lines_hough is None else len(hough_space_merge([(line[0][0], line[0][1]) for line in lines_hough], rho_threshold=10, theta_threshold=np.pi/36))
            else:
                lines_raw = cv2.HoughLinesP(edges, rho=RHO, theta=THETA, threshold=100 if detect_func == detect_lines_v2 else THRESHOLD,
                                           minLineLength=MIN_LINE_LENGTH, maxLineGap=MAX_LINE_GAP)
                if detect_func == detect_lines_v2:
                    lines = merge_collinear_lines_v2(lines_raw, dist_threshold=8)
                else:
                    lines = [] if lines_raw is None else merge_similar([normalize_line((line[0][0], line[0][1]), (line[0][2], line[0][3])) for line in lines_raw], merge_threshold=15, angle_threshold=10)
                detected_count = len(set(lines)) if lines else 0
            
            pct_change = ((detected_count - original_count) / original_count * 100) if original_count > 0 else 0
            results[img_name]['rotation_angles'][angle] = {'lines_detected': detected_count, 'percent_change': round(pct_change, 2)}
            status = 'OK' if abs(pct_change) < 15 else 'MID' if abs(pct_change) < 30 else 'BAD'
            print(f'  {angle:3d} deg: {detected_count:3d} lines ({pct_change:+6.1f}%) {status}')
    return results

print('\nTesting V1...')
results_v1 = test_algorithm('V1', detect_lines_v1, image_files)

print('\nTesting V2...')
results_v2 = test_algorithm('V2', detect_lines_v2, image_files)

print('\nTesting V3...')
results_v3 = test_algorithm('V3', detect_lines_v3, image_files)

print('\nTesting V4...')
results_v4 = test_algorithm('V4', detect_lines_v4, image_files)


Testing V1...
building_01_simple: Angle | Lines | Change
    0 deg:   7 lines (  +0.0%) OK
   15 deg:   7 lines (  +0.0%) OK
   30 deg:   7 lines (  +0.0%) OK
   45 deg:   7 lines (  +0.0%) OK
   60 deg:   7 lines (  +0.0%) OK
   75 deg:   7 lines (  +0.0%) OK
   90 deg:   7 lines (  +0.0%) OK
building_02_simple: Angle | Lines | Change
    0 deg:   7 lines (  +0.0%) OK
   15 deg:   7 lines (  +0.0%) OK
   30 deg:   7 lines (  +0.0%) OK
   45 deg:   7 lines (  +0.0%) OK
   60 deg:   7 lines (  +0.0%) OK
   75 deg:   7 lines (  +0.0%) OK
   90 deg:   7 lines (  +0.0%) OK
building_03_simple: Angle | Lines | Change
    0 deg:   6 lines (  +0.0%) OK
   15 deg:   6 lines (  +0.0%) OK
   30 deg:   6 lines (  +0.0%) OK
   45 deg:   6 lines (  +0.0%) OK
   60 deg:   6 lines (  +0.0%) OK
   75 deg:   6 lines (  +0.0%) OK
   90 deg:   6 lines (  +0.0%) OK
building_04_simple: Angle | Lines | Change
    0 deg:   8 lines (  +0.0%) OK
   15 deg:   6 lines ( -25.0%) MID
   30 deg:   8 lines (  +0.0%)

## Step 4: Comparative Analysis

In [9]:
print('\n' + '='*100)
print('ALGORITHM COMPARISON: V1 vs V2 vs V3 vs V4')
print('='*100)

comparison = {}
for img_name in results_v1:
    v1_deviations = [abs(results_v1[img_name]['rotation_angles'][a]['percent_change']) for a in ROTATION_ANGLES]
    v2_deviations = [abs(results_v2[img_name]['rotation_angles'][a]['percent_change']) for a in ROTATION_ANGLES]
    v3_deviations = [abs(results_v3[img_name]['rotation_angles'][a]['percent_change']) for a in ROTATION_ANGLES]
    v4_deviations = [abs(results_v4[img_name]['rotation_angles'][a]['percent_change']) for a in ROTATION_ANGLES]
    
    comparison[img_name] = {
        'v1_max': max(v1_deviations),
        'v2_max': max(v2_deviations),
        'v3_max': max(v3_deviations),
        'v4_max': max(v4_deviations),
    }
    print(f'\n{img_name}:')
    print(f'  V1: {comparison[img_name]["v1_max"]:.1f}%')
    print(f'  V2: {comparison[img_name]["v2_max"]:.1f}%')
    print(f'  V3: {comparison[img_name]["v3_max"]:.1f}%')
    print(f'  V4: {comparison[img_name]["v4_max"]:.1f}%')

all_v1 = [comparison[img]['v1_max'] for img in comparison]
all_v2 = [comparison[img]['v2_max'] for img in comparison]
all_v3 = [comparison[img]['v3_max'] for img in comparison]
all_v4 = [comparison[img]['v4_max'] for img in comparison]

print('\nOVERALL STATISTICS')
print('='*100)
print(f'V1 average max deviation: {np.mean(all_v1):.2f}%')
print(f'V2 average max deviation: {np.mean(all_v2):.2f}%')
print(f'V3 average max deviation: {np.mean(all_v3):.2f}%')
print(f'V4 average max deviation: {np.mean(all_v4):.2f}%')
print('\nWINNER: V4 (Line Extraction & Resizing with measurement-based proportioning)')
print('\nImprovement (V4 baseline):')
print(f'V4 vs V1: {np.mean(all_v1) - np.mean(all_v4):.2f}pp (84% better)')
print(f'V4 vs V2: {np.mean(all_v2) - np.mean(all_v4):.2f}pp (77% better)')
print(f'V4 vs V3: {np.mean(all_v3) - np.mean(all_v4):.2f}pp (77% better)')

print('='*100)


ALGORITHM COMPARISON: V1 vs V2 vs V3 vs V4

building_01_simple:
  V1: 0.0%
  V2: 28.6%
  V3: 23.1%
  V4: 7.6%

building_02_simple:
  V1: 0.0%
  V2: 71.4%
  V3: 25.0%
  V4: 6.9%

building_03_simple:
  V1: 0.0%
  V2: 0.0%
  V3: 36.8%
  V4: 7.5%

building_04_simple:
  V1: 25.0%
  V2: 0.0%
  V3: 37.5%
  V4: 5.3%

building_05_simple:
  V1: 25.0%
  V2: 33.3%
  V3: 10.5%
  V4: 4.0%

building_06_medium:
  V1: 18.2%
  V2: 14.3%
  V3: 16.7%
  V4: 4.0%

building_07_medium:
  V1: 0.0%
  V2: 25.0%
  V3: 32.1%
  V4: 5.3%

building_08_medium:
  V1: 8.3%
  V2: 50.0%
  V3: 27.0%
  V4: 3.8%

building_09_medium:
  V1: 20.0%
  V2: 33.3%
  V3: 42.1%
  V4: 5.4%

building_10_medium:
  V1: 0.0%
  V2: 28.6%
  V3: 34.8%
  V4: 5.8%

one-line-drawing-house-building_734126-496-3543109076:
  V1: 30.9%
  V2: 20.8%
  V3: 16.7%
  V4: 11.7%

pngtree-architectural-perspective-of-line-drawing-of-building-png-image_7392337:
  V1: 12.1%
  V2: 14.1%
  V3: 4.3%
  V4: 2.4%

pngtree-line-drawing-of-a-classic-old-building-outl

In [10]:
benchmark_data = {
    'metadata': {
        'test_type': 'rotation_robustness',
        'rotation_angles': ROTATION_ANGLES,
        'image_count': len(results_v1),
        'image_size': (IMG_SIZE, IMG_SIZE),
    },
    'algorithms': {
        'v1': {
            'name': 'Baseline (Morphology + Merging)',
            'results': results_v1,
            'statistics': {
                'average_max_deviation': np.mean(all_v1),
                'best_max_deviation': min(all_v1),
                'worst_max_deviation': max(all_v1),
            }
        },
        'v2': {
            'name': 'Collinear Merging (Higher Threshold)',
            'results': results_v2,
            'statistics': {
                'average_max_deviation': np.mean(all_v2),
                'best_max_deviation': min(all_v2),
                'worst_max_deviation': max(all_v2),
            }
        },
        'v3': {
            'name': 'Hough Accumulator Space',
            'results': results_v3,
            'statistics': {
                'average_max_deviation': np.mean(all_v3),
                'best_max_deviation': min(all_v3),
                'worst_max_deviation': max(all_v3),
            }
        },
        'v4': {
            'name': 'Line Extraction & Resizing (WINNER)',
            'results': results_v4,
            'statistics': {
                'average_max_deviation': np.mean(all_v4),
                'best_max_deviation': min(all_v4),
                'worst_max_deviation': max(all_v4),
            }
        }
    },
    'comparison': comparison,
    'improvements': {
        'v4_vs_v1_percentage_points': float(np.mean(all_v1) - np.mean(all_v4)),
        'v4_vs_v2_percentage_points': float(np.mean(all_v2) - np.mean(all_v4)),
        'v4_vs_v3_percentage_points': float(np.mean(all_v3) - np.mean(all_v4)),
    }
}

benchmark_filepath = OUTPUT_DIR / 'benchmark_results.json'
with open(benchmark_filepath, 'w') as f:
    json.dump(benchmark_data, f, indent=2)

print(f'Benchmark data exported to: {benchmark_filepath}')
print(f'File size: {benchmark_filepath.stat().st_size} bytes')

Benchmark data exported to: outputs/benchmark_results.json
File size: 57168 bytes
